# Drex selectors for LlamaIndex

Route queries with a Drex decision model instead of an LLM. Drex scores the options and returns probabilities; it writes no text.

Install (coming soon to PyPI; from source meanwhile):

```
pip install "git+https://github.com/nace-ai/llama-index-nace#subdirectory=packages/llama-index-selectors-nace"
```

Set `NACE_API_KEY` in the environment before running.

The default model is `drex-max-v1.1`.

In [1]:
from llama_index.core.tools import ToolMetadata
from llama_index.selectors.nace import NaceMultiSelector, NaceSingleSelector

tools = [
    ToolMetadata(name="pricing", description="Plans, prices, invoices and refunds"),
    ToolMetadata(name="docs", description="API reference and error codes"),
    ToolMetadata(name="status", description="Current outages and incident history"),
]

## Single selection

`select` is sync and works inside a notebook's running event loop.

In [2]:
result = NaceSingleSelector().select(tools, "Why do I keep getting 429 errors from the API?")
print(tools[result.selections[0].index].name, result.reason)
print(result.probabilities)
print(result.model, result.usage)

docs p=0.95
{'pricing': 0.03667574410024792, 'docs': 0.9509328096260579, 'status': 0.012391446273694246}
drex-max-v1.1 input_tokens=123 output_tokens=114 image_tokens=None


## Confidence guard

With `min_confidence`, an unsure answer falls back to `default_index`, or raises `NaceLowConfidenceError` when there is none.

In [3]:
selector = NaceSingleSelector(min_confidence=0.8, default_index=0)
result = selector.select(tools, "Thanks!")
print(tools[result.selections[0].index].name, "| fell_back =", result.fell_back)
print(result.reason)

pricing | fell_back = True
p=0.36 below min_confidence=0.8; used default_index=0


## Multi selection

One yes/no question per tool, all in a single request. Tools at or above `threshold` are returned, best first. If none passes, the top tool is returned with `fell_back=True`, so the result is never empty. Each tool is one question, so Drex's per-request question limit caps the tool count.

In [4]:
multi = NaceMultiSelector(threshold=0.3, max_outputs=2)
result = multi.select(tools, "Why is the API down, and will I be refunded for the outage?")
print([tools[s.index].name for s in result.selections])
print(result.probabilities, "fell_back =", result.fell_back)

['status', 'pricing']
{'pricing': 0.8748077750205994, 'docs': 0.04417244799756348, 'status': 0.9852956469198041} fell_back = False


## Workflows branching

A selector is a good branch point in a `Workflow`: `NaceLowConfidenceError` becomes a human-review branch. No LLM is called.

In [5]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.selectors.nace import NaceLowConfidenceError

CHOICES = [
    "billing: invoices, refunds, charges, subscriptions and payment methods",
    "technical: errors, outages, API usage, integrations and bugs",
]


class Routed(Event):
    route: str
    probabilities: dict[str, float]


class Triage(Workflow):
    selector = NaceSingleSelector(min_confidence=0.6)

    @step
    async def classify(self, ev: StartEvent) -> Routed:
        try:
            result = await self.selector.aselect(CHOICES, ev.message)
        except NaceLowConfidenceError as error:
            return Routed(route="human_review", probabilities=getattr(error, "probabilities", {}))
        route = ["billing", "technical"][result.selections[0].index]
        return Routed(route=route, probabilities=result.probabilities)

    @step
    async def finish(self, ev: Routed) -> StopEvent:
        return StopEvent(result={"route": ev.route, "probabilities": ev.probabilities})


for message in ["I was charged twice for my March invoice.", "Thanks!"]:
    print(await Triage(timeout=60).run(message=message))

{'route': 'billing', 'probabilities': {'option_1': 0.9946329074916962, 'option_2': 0.005367092508303765}}


{'route': 'human_review', 'probabilities': {'option_1': 0.5225909948348999, 'option_2': 0.4774090051651001}}


## With RouterQueryEngine

`RouterQueryEngine` resolves `Settings.llm` (OpenAI by default) when it is built, even if it only routes, so pass an LLM. `MockLLM` is used here to stay key-free. In a real app pass the LLM that writes answers: Drex replaces the routing decision, not answer generation (and with multi-select, the LLM also merges the answers).

In [6]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.embeddings import MockEmbedding
from llama_index.core.llms import MockLLM
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool

llm = MockLLM()
embed = MockEmbedding(embed_dim=8)


def tool(m: ToolMetadata, text: str) -> QueryEngineTool:
    index = VectorStoreIndex.from_documents([Document(text=text)], embed_model=embed)
    return QueryEngineTool(query_engine=index.as_query_engine(llm=llm), metadata=m)


engine_tools = [
    tool(tools[0], "The Pro plan costs $20."),
    tool(tools[1], "A 429 means you hit the rate limit."),
    tool(tools[2], "No incidents today."),
]
router = RouterQueryEngine(selector=NaceSingleSelector(), query_engine_tools=engine_tools, llm=llm)
response = router.query("What does a 429 mean?")
selected = response.metadata["selector_result"]
print([engine_tools[s.index].metadata.name for s in selected.selections])
print(selected.probabilities)

['docs']
{'pricing': 0.0029151812623386315, 'docs': 0.9946519282327666, 'status': 0.002432890504894802}


## Known limitation of multi-select

The default multi-select question asks whether each option is *needed*, so it is conservative on purely definitional questions: a docs tool scored about 0.27 for "What does 429 mean?" in one probe, below the 0.5 default, and the selector fell back to the top option. Lower `threshold`, or pass your own `instructions` (`{label}` and `{description}` are filled in).

In [7]:
custom = NaceMultiSelector(
    instructions="Could the option below help answer the query? Option {label}: {description}",
    threshold=0.4,
)
result = custom.select(tools, "What does 429 mean?")
print([tools[s.index].name for s in result.selections], result.probabilities, result.fell_back)

['docs'] {'pricing': 0.0112499861733526, 'docs': 0.9876984137066063, 'status': 0.03118372504869257} False
